# Análise de Indicadores Econômicos e Mercado Financeiro Brasileiro

**Autor:** Maycon Serzedelo  
**Objetivo:** Análise macroeconômica do Brasil + comparação com mercados emergentes + previsão + regimes de volatilidade.

---

## 1. Configuração e Importações

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import carregar_dados

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento dos Dados

In [ ]:
df = carregar_dados(data_inicio='2018-01-01')
print('\nShape:', df.shape)
df.tail()

## 3. Evolução Temporal dos Indicadores Brasileiros

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
fig.suptitle('Indicadores Econômicos do Brasil', fontsize=16, fontweight='bold')

indicadores = [
    ('IPCA', 'IPCA'), ('IGP_M', 'IGP-M'), ('Selic', 'Selic'),
    ('Cambio_USD_BRL', 'Câmbio'), ('IBC_Br', 'IBC-Br'), ('PIB', 'PIB'),
    ('Producao_Industrial', 'Produção Industrial'), ('Vendas_Varejo', 'Vendas Varejo'),
    ('Desemprego', 'Desemprego'), ('Credito_Total', 'Crédito'),
    ('Reservas_Internacionais', 'Reservas'), ('Ibovespa', 'Ibovespa')
]

for ax, (col, titulo) in zip(axes.flat, indicadores):
    if col in df.columns:
        df[col].plot(ax=ax, title=titulo)
        ax.set_xlabel('')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Comparação com Outros Mercados Emergentes

Comparamos o **Ibovespa** com:
- **S&P 500** (EUA – referência global)
- **IPC México**
- **Nifty 50** (Índia)
- **Shanghai Composite** (China)

In [ ]:
# Normaliza os índices para base 100 no início do período
mercados = ['Ibovespa', 'SP500', 'Mexico_IPC', 'India_Nifty', 'China_Shanghai']
mercados_existentes = [m for m in mercados if m in df.columns]

df_norm = df[mercados_existentes].dropna(how='all')
df_norm = df_norm / df_norm.iloc[0] * 100

plt.figure(figsize=(14, 6))
for col in df_norm.columns:
    plt.plot(df_norm.index, df_norm[col], label=col, linewidth=1.5)

plt.title('Desempenho Comparado – Base 100', fontsize=14, fontweight='bold')
plt.ylabel('Base 100')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Retornos acumulados e volatilidade
retornos = df[mercados_existentes].pct_change()

resumo = pd.DataFrame({
    'Retorno Total (%)': (df[mercados_existentes].iloc[-1] / df[mercados_existentes].iloc[0] - 1) * 100,
    'Volatilidade Anualizada (%)': retornos.std() * np.sqrt(252) * 100,
    'Sharpe (aprox)': (retornos.mean() * 252) / (retornos.std() * np.sqrt(252))
}).round(2)

print('=== Comparação de Performance ===')
print(resumo.sort_values('Retorno Total (%)', ascending=False))

In [ ]:
# Matriz de correlação entre os mercados
plt.figure(figsize=(8, 6))
corr_mercados = df[mercados_existentes].pct_change().corr()
sns.heatmap(corr_mercados, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Correlação de Retornos entre Mercados', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Matriz de Correlação – Indicadores do Brasil

In [ ]:
cols_br = ['IPCA', 'IGP_M', 'Selic', 'Cambio_USD_BRL', 'IBC_Br', 'PIB',
           'Producao_Industrial', 'Vendas_Varejo', 'Desemprego', 'Credito_Total',
           'Reservas_Internacionais', 'Ibovespa']
cols_br = [c for c in cols_br if c in df.columns]

plt.figure(figsize=(12, 10))
sns.heatmap(df[cols_br].corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlação entre Indicadores Brasileiros', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Análise de Regimes de Volatilidade (Ibovespa)

In [ ]:
df['Retorno'] = df['Ibovespa'].pct_change()
df['Vol_21d'] = df['Retorno'].rolling(21).std() * np.sqrt(252)
mediana_vol = df['Vol_21d'].median()
df['Regime'] = np.where(df['Vol_21d'] > mediana_vol, 'Alta Volatilidade', 'Baixa Volatilidade')

print(f'Mediana da volatilidade anualizada: {mediana_vol:.2%}')
print(df['Regime'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for regime, cor in [('Baixa Volatilidade', 'green'), ('Alta Volatilidade', 'red')]:
    mask = df['Regime'] == regime
    axes[0].plot(df.index[mask], df['Ibovespa'][mask], '.', color=cor, label=regime, markersize=2)

axes[0].set_title('Ibovespa por Regime de Volatilidade', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df['Vol_21d'], color='steelblue')
axes[1].axhline(mediana_vol, color='red', linestyle='--', label=f'Mediana ({mediana_vol:.1%})')
axes[1].set_title('Volatilidade Móvel 21 dias (anualizada)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Modelos Simples de Previsão (Ibovespa)

In [ ]:
serie = df['Ibovespa'].dropna()
horizonte = 60
treino = serie.iloc[:-horizonte]
teste = serie.iloc[-horizonte:]

print(f'Treino: {len(treino)} pontos | Teste: {len(teste)} pontos')

In [ ]:
# ARIMA
modelo_arima = ARIMA(treino, order=(2, 1, 2)).fit()
previsao_arima = modelo_arima.forecast(steps=horizonte)
previsao_arima.index = teste.index

# Prophet
df_prophet = treino.reset_index()
df_prophet.columns = ['ds', 'y']
modelo_prophet = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
modelo_prophet.fit(df_prophet)
futuro = modelo_prophet.make_future_dataframe(periods=horizonte, freq='B')
forecast = modelo_prophet.predict(futuro)
pred_prophet = forecast.set_index('ds')['yhat'].reindex(teste.index)

# Métricas
print('=== Performance ===')
print(f'ARIMA   → MAE: {mean_absolute_error(teste, previsao_arima):,.0f} | RMSE: {np.sqrt(mean_squared_error(teste, previsao_arima)):,.0f}')
print(f'Prophet → MAE: {mean_absolute_error(teste, pred_prophet):,.0f} | RMSE: {np.sqrt(mean_squared_error(teste, pred_prophet)):,.0f}')

## 8. Insights Finais

- O Ibovespa costuma ter maior volatilidade que mercados desenvolvidos (S&P 500).
- Correlação com outros emergentes varia ao longo do tempo (em crises sobe).
- Regimes de alta volatilidade no Brasil geralmente coincidem com estresse cambial ou de juros.
- Modelos de série temporal pura têm poder preditivo limitado no curto prazo para ações.

**Próximos passos possíveis:** dashboard Streamlit, GARCH, variáveis exógenas nos modelos.